In [ ]:
"""
准备好 所有的pdb文件，同时修改好晶胞大小，和pdb文件名。
"""
import os
import argparse
from ase.io import read,write
from glob import glob
import subprocess
import numpy as np
from ase.optimize import LBFGS
from ase.filters import UnitCellFilter
template ="""

structure {pdb_file}
  number {number}
  inside box 0. 0. 0. {l1} {l2} {l3}
end structure

"""

head_text = """
tolerance 2.0
add_box_sides 2.0
filetype pdb
output packmol.pdb

"""
structure = "structure.pdb"
molecules = {
"N2.pdb": 24,
"CO2.pdb": 12,
"CO.pdb":12,
"H2O.pdb":12,
}

# stru = read(structure)
# cell_length = np.power(stru.get_volume(),1/3)
cell_length = 10
cell = [cell_length, cell_length, cell_length]
cycle_text = "\n".join([template.format(pdb_file = f"{key}", number = value, l1 = cell[0], l2 = cell[1], l3 = cell[2]) for key,value in molecules.items()])
text = head_text + cycle_text

with open("packmol.inp",'w') as file:
    file.write(text)

# 执行 Packmol（使用 subprocess 替代 os.system）
# subprocess.run(["/home/liuzf/soft/packmol/packmol", "<", "packmol.inp"], check=True)
os.system("/home/liuzf/soft/packmol/packmol < packmol.inp")

# 读取生成的 packmol.pdb 文件
atoms = read("packmol.pdb")

# 使用元素符号排序原子
elements = atoms.get_chemical_symbols()
sorted_atoms = atoms[[i for i in sorted(range(len(elements)), key=lambda x: elements[x])]]

from mattersim.forcefield import MatterSimCalculator
calculator=MatterSimCalculator(load_path="mattersim-v1.0.0-1m",device="cuda")
atoms.calc = calculator
ucf = UnitCellFilter(atoms,hydrostatic_strain=True)
opt = LBFGS(ucf)
opt.run(fmax=0.05,steps=200)



################################################################################

 PACKMOL - Packing optimization for the automated generation of
 starting configurations for molecular dynamics simulations.
 
                                                              Version 20.15.3 

################################################################################

  Packmol must be run with: packmol < inputfile.inp 

  Userguide at: http://m3g.iqm.unicamp.br/packmol 

  Reading input file... (Control-C aborts)
  Types of coordinate files specified: pdb
  Will print BOX SIDE informations. 
  Will sum    2.0000000000000000       to each side length on print
  Seed for random number generator:      1234567
  Output file: packmol.pdb
  Reading coordinate file: N2.pdb
  Reading coordinate file: CO2.pdb
  Reading coordinate file: CO.pdb
  Reading coordinate file: H2O.pdb
  Number of independent structures:            4
  The structures are: 
  Structure            1 :N2.pdb(           2

In [14]:
atoms.get_cell()

Cell([15.397275439901282, 15.358910135652211, 15.358910135652211])

In [20]:
from ase.build.supercells import make_supercell
# 读取生成的 packmol.pdb 文件
atoms = read("packmol.pdb")
matrix = [[2,0,0],[0,2,0],[0,0,2]]
atoms = make_supercell(atoms,matrix)
# 写入 VASP POSCAR 文件
elements = atoms.get_chemical_symbols()

# 使用元素符号排序原子
sorted_atoms = atoms[[i for i in sorted(range(len(elements)), key=lambda x: elements[x])]]

# 写入重新排序后的结构
# write("sorted_output.pdb", sorted_atoms)
write("POSCAR", atoms)